# 02 — Features: implicit interaction matrix + cert-content TF-IDF

**Project:** Micro-Cert Recommender (H07)

Two design objects feed the two-tower recommender: a sparse implicit interaction matrix (collaborative signal) and a TF-IDF over `skills_taught` (content signal). Build, audit, and persist them as the inputs the model fitter consumes.

In [ ]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sys.path.insert(0, str(Path.cwd().parent / 'src'))
from microcert_rec.data import PROCESSED, load_all, make_training_artifacts
from microcert_rec.features import build_interaction_matrix, fit_cert_tfidf, learner_text

sns.set_theme(style='whitegrid', context='notebook')
plt.rcParams['figure.dpi'] = 110

In [ ]:
if (PROCESSED / 'learners.parquet').exists():
    learners, certs, interactions = load_all()
else:
    learners, certs, interactions = make_training_artifacts()

## 1. Build the implicit interaction matrix

In [ ]:
R, learner_ids, cert_ids = build_interaction_matrix(learners, certs, interactions)
print('R shape:', R.shape, 'nnz:', R.nnz, 'density:', R.nnz / (R.shape[0] * R.shape[1]))

## 2. Per-row + per-column weight

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 3.4))
axes[0].hist(np.asarray(R.sum(axis=1)).ravel(), bins=30, color='#3a7ca5', edgecolor='white')
axes[0].set_title('per-learner summed weight')
axes[1].hist(np.asarray(R.sum(axis=0)).ravel(), bins=30, color='#9c6644', edgecolor='white')
axes[1].set_title('per-cert summed weight')
plt.tight_layout(); plt.show()

## 3. Fit the cert-content TF-IDF over `skills_taught`

In [ ]:
vec, X_certs = fit_cert_tfidf(certs)
print('cert content matrix shape:', X_certs.shape)
print('vocab size:', len(vec.vocabulary_))

## 4. Cert-cert content nearest neighbours

In [ ]:
from sklearn.metrics.pairwise import cosine_similarity
S = cosine_similarity(X_certs)
for cert_id in certs['cert_id'].sample(3, random_state=0):
    i = certs.index[certs['cert_id'] == cert_id][0]
    nn = np.argsort(-S[i])[1:4]
    print(f'{certs.iloc[i]["title"]} -> {[certs.iloc[j]["title"] for j in nn]}')

## 5. Learner-text encoding

`learner_text(skill_set)` renders an active learner's skill set into the same vocabulary space as `skills_taught`. We exercise it on a sample learner.

In [ ]:
skill_cols = [c for c in learners.columns if c.startswith('skill__')]
row = learners.iloc[0]
skills = [c.replace('skill__', '') for c in skill_cols if row[c] == 1]
txt = learner_text(skills)
print(txt)
qv = vec.transform([txt]).toarray()
print('learner query vec L2 norm:', float(np.linalg.norm(qv)))

## 6. Content-side learner→cert top scores

In [ ]:
scores = cosine_similarity(qv, X_certs).ravel()
top = np.argsort(-scores)[:10]
certs.iloc[top][['cert_id', 'title', 'theme', 'hours']].assign(score=scores[top].round(3))

## 7. Hand-off

These two design objects are passed into `microcert_rec.models.fit` which trains a TruncatedSVD over `R` and stores both towers in `models/two_tower.joblib`.